# Flow Matching: обучить поле скоростей

**Flow Matching for Generative Modeling** · Lipman et al. · 2022

[Оригинальная статья](https://arxiv.org/abs/2210.02747)

**Цель урока:** Обучить conditional velocity objective и получить выборки численным решением ODE.

## Перед началом

Производная, MSE, Gaussian samples; вводный VLA.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

Flow matching учит поле скоростей, которое переносит начальное распределение в распределение данных. На training можно выбрать точки пути и их целевые скорости, не решая ODE на каждом шаге. На sampling интегрируем обученное поле.

Для action policies поле дополнительно зависит от наблюдения и инструкции. Здесь conditioning отсутствует: разбираем одномерное распределение. Берём независимые пары noise/data и линейную интерполяцию с нулевой конечной дисперсией. Это частный conditional path, а не решение глобальной optimal transport задачи.

## Граница воспроизведения

Простой linear conditional flow matching. Нет полного π0/SmolVLA, vision encoder, робота и benchmark статьи.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`x_t = (1-t)*x0 + t*x1`; target velocity=`x1-x0`; `L = E[||v_theta(x_t,t) - (x1-x0)||²]`.

В каждой паре скорость постоянна, но conditional paths пересекаются: оптимальная сеть предсказывает условное среднее velocity по доступным x_t,t. На sampling: `x ← x + dt*v_theta(x,t)`, t растёт от 0 до 1. Обратное направление требует согласованно поменять знак/определение времени.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Проверяем endpoints и производную линейного пути через finite differences.

In [ ]:
x0 = torch.randn(32, 1); x1 = 2.0 + 0.5 * torch.randn(32, 1)
def path(t):
    return (1 - t) * x0 + t * x1
assert torch.equal(path(0), x0) and torch.equal(path(1), x1)
dt = 1e-3
derivative = (path(0.4 + dt) - path(0.4)) / dt
assert torch.allclose(derivative, x1 - x0, atol=3e-4)
print('endpoint means:', x0.mean().item(), x1.mean().item())

## Шаг 2

Сеть видит x_t и t, а не пару endpoints. Это отличие training target от входа модели.

In [ ]:
velocity = nn.Sequential(nn.Linear(2, 48), nn.SiLU(), nn.Linear(48, 48), nn.SiLU(), nn.Linear(48, 1))
optimizer = torch.optim.Adam(velocity.parameters(), lr=0.003)
history = []
for step in range(300):
    base = torch.randn(128, 1); data = 2.0 + 0.5 * torch.randn(128, 1)
    t = torch.rand(128, 1); point = (1 - t) * base + t * data
    loss = F.mse_loss(velocity(torch.cat([point, t], -1)), data - base)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
assert math.isfinite(history[-1])
plt.plot(history, color='#d73535', alpha=0.6); plt.xlabel('step'); plt.ylabel('velocity MSE'); plt.show()

## Шаг 3

Сравниваем шаги Euler при одинаковом начальном шуме. Различие отражает discretization и ошибку выученного поля.

In [ ]:
initial = torch.randn(1000, 1)
def sample(steps):
    x = initial.clone()
    with torch.no_grad():
        for index in range(steps):
            t = torch.full_like(x, index / steps)
            x = x + velocity(torch.cat([x, t], -1)) / steps
    return x
coarse, fine = sample(5), sample(50)
assert torch.isfinite(fine).all()
print('Euler 5/50 means:', coarse.mean().item(), fine.mean().item(), 'target mean: 2')
print('fine std:', fine.std().item(), 'target std: 0.5')
plt.hist(fine.flatten(), bins=40, density=True, alpha=0.6, color='#d73535', label='generated')
plt.hist((2 + 0.5 * torch.randn(1000)).flatten(), bins=40, density=True, alpha=0.3, color='gray', label='target')
plt.legend(); plt.show()

## Как читать результат

Training MSE не обязан идти к нулю: для одного x_t,t возможны разные пары и velocities. Больше sampling steps уменьшает discretization error, но не исправляет ошибочное поле. Для VLA нужны ещё conditioning, action normalization и dataset-specific masks.

## Самостоятельные эксперименты

1. Поменяйте знак velocity на sampling.
2. Сравните 1, 5, 20, 100 шагов при одном seed.
3. Добавьте бинарное conditioning: target mean -2 или +2.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Неправильный знак переносит распределение не в ту сторону. Сравнивать нужно одинаковые initial samples. Conditional label должен присутствовать и при training, и при sampling; иначе сеть смешивает две задачи.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.